# Re-plot Experiment 5 figures

Standalone notebook to **re-plot all Exp 5 figures from the saved JSON** in `results/`,
without re-running the big training notebook. Tweak styling here and re-run the cell.

Data sources (everything needed is already on disk):

| Figure | Source JSON |
|---|---|
| `exp5_gradient_credit`, `exp5_credit_summary` | `results/exp5_gradient_credit.json` |
| `exp5_learning_curves` | `results/exp5_learning_curves.json` (`curves`) |
| `exp5_speed_threshold`, `exp5_speed_intervals` | `results/exp5_learning_curves.json` (`per_seed`) |

The last cell commits the regenerated figures and pushes to GitHub.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os, json, subprocess
import numpy as np
import matplotlib.pyplot as plt

# Make sure we run from the repo root so `results/` and `experiments/` resolve.
# Works locally (already in the repo or a subfolder) and on Colab (fresh session).
REPO_URL = "https://github.com/simonpeter02/e-prop-in-deep-networks.git"
REPO_DIR = "e-prop-in-deep-networks"

def _has_results():
    return os.path.isfile("results/exp5_learning_curves.json")

if not _has_results():
    if os.path.isfile(os.path.join("..", "results", "exp5_learning_curves.json")):
        os.chdir("..")                       # launched from a subfolder
    elif os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)                    # repo already cloned this session
    else:
        print("Cloning repo ...")            # fresh Colab session
        subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_DIR)

assert _has_results(), f"Could not find results/ JSON — cwd={os.getcwd()}"
print("cwd:", os.getcwd())

RESULTS = "results"

def save_fig(fig, name, out_dir=RESULTS):
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(f"{out_dir}/{name}.pdf", bbox_inches="tight")
    fig.savefig(f"{out_dir}/{name}.svg", bbox_inches="tight")
    print(f"Saved {out_dir}/{name}.pdf / .svg")

def load(name):
    with open(f"{RESULTS}/{name}.json") as f:
        return json.load(f)

# Shared style (matches the big notebook)
col = {"bptt": "k", "full": "C0", "ablate_temporal": "C3", "ablate_spatial": "C1"}
e5_methods = [("bptt", "BPTT"), ("full", "deep e-prop (full)"),
              ("ablate_temporal", "ablate temporal"), ("ablate_spatial", "ablate spatial")]
_lab = {m: lab for m, lab in e5_methods}

# Config defaults (from the big notebook — edit freely)
E5_DELAY_MAIN = 12     # delay shown in the credit-summary bars
E5_THETA = 0.9         # accuracy threshold for the speed figure


## 5.1  Gradient credit + credit summary
Source: `exp5_gradient_credit.json`

In [ ]:
# ── 5.1  Per-layer cosine vs delay (+ cross-temporal share) and summary bars ──
gc = load("exp5_gradient_credit")
DELAYS = gc["delays"]
e5_mean, e5_sem = gc["mean"], gc["sem"]   # keyed by str(delay) -> {full_low, ...}

def _ser(stat, k):
    return np.nan_to_num(np.array([stat[str(d)][k] for d in DELAYS], dtype=float))

# Figure 1 — per-layer cosine vs delay, with stderr bands
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
for k, c, ls, lab, mfc in [("full_up", "C0", ":", "full · output-adjacent", "none"),
                           ("full_low", "C0", "-", "full · input-adjacent", "C0"),
                           ("temp_low", "C3", "-", "ablate_temporal · input-adjacent", "C3"),
                           ("spat_low", "C1", "-", "ablate_spatial · input-adjacent (=0)", "C1")]:
    mu, er = _ser(e5_mean, k), _ser(e5_sem, k)
    ax.plot(DELAYS, mu, marker="o", color=c, ls=ls, label=lab, markerfacecolor=mfc)
    ax.fill_between(DELAYS, mu - er, mu + er, color=c, alpha=0.15)
ax.axhline(0, color="gray", ls=":"); ax.set_xlabel("delay D"); ax.set_ylabel("cosine vs BPTT")
ax.set_ylim(-0.1, 1.05); ax.legend(fontsize=8)
ax = axes[1]
mu, er = _ser(e5_mean, "xshare"), _ser(e5_sem, "xshare")
ax.plot(DELAYS, mu, "o-", color="C2"); ax.fill_between(DELAYS, mu - er, mu + er, color="C2", alpha=0.15)
ax.set_xlabel("delay D"); ax.set_ylabel("||full - ablate_temporal|| / ||full||"); ax.set_ylim(0, 1.05)
fig.tight_layout(); save_fig(fig, "exp5_gradient_credit"); plt.show()

# Figure 2 — summary bars at the main delay
db = E5_DELAY_MAIN if str(E5_DELAY_MAIN) in e5_mean else DELAYS[len(DELAYS) // 2]
kf = {("full", "low"): "full_low", ("full", "top"): "full_up",
      ("ablate_temporal", "low"): "temp_low", ("ablate_temporal", "top"): "temp_up",
      ("ablate_spatial", "low"): "spat_low", ("ablate_spatial", "top"): "spat_up"}
fig, ax = plt.subplots(figsize=(7, 4.5)); x = np.arange(2); w = 0.26
for i, (meth, c) in enumerate([("full", "C0"), ("ablate_temporal", "C3"), ("ablate_spatial", "C1")]):
    mus = [np.nan_to_num(e5_mean[str(db)][kf[(meth, l)]]) for l in ("low", "top")]
    ers = [np.nan_to_num(e5_sem[str(db)][kf[(meth, l)]]) for l in ("low", "top")]
    ax.bar(x + (i - 1) * w, mus, w, yerr=ers, capsize=4, color=c, label=meth.replace("ablate_", "ablate "))
ax.set_xticks(x); ax.set_xticklabels(["input-adjacent", "output-adjacent"]); ax.set_ylim(-0.05, 1.05)
ax.axhline(0, color="gray", ls=":"); ax.set_ylabel("cosine vs BPTT")
ax.legend(fontsize=8); fig.tight_layout(); save_fig(fig, "exp5_credit_summary"); plt.show()


## 5.2  Adam learning curves
Source: `exp5_learning_curves.json` (`curves`)

In [ ]:
# ── 5.2  Learning curves — equal final accuracy, credit quality sets speed ────
lc = load("exp5_learning_curves")
E5_EVAL = lc["eval_every"]
E5_LR = lc["lr"]
e5_curves = {m: (np.array(v[0]), np.array(v[1])) for m, v in lc["curves"].items()}

fig, ax = plt.subplots(figsize=(7, 4.5))
for meth, lab in e5_methods:
    mu, er = e5_curves[meth]
    x = np.arange(len(mu)) * E5_EVAL
    ax.plot(x, mu, "-o", ms=3, color=col[meth], label=lab)
    ax.fill_between(x, mu - er, mu + er, color=col[meth], alpha=0.15)
ax.axhline(0.5, color="gray", ls="--", label="chance")
ax.set_xlabel("training step"); ax.set_ylabel("accuracy (held-out)"); ax.set_ylim(0.45, 1.02)
ax.set_title(f"Adam (lr={E5_LR:.0e}): equal final accuracy, credit quality sets speed")
ax.legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "exp5_learning_curves"); plt.show()


## 5.3  Speed significance figures
Source: `exp5_learning_curves.json` (`per_seed`). Re-runs the paired/cluster-permutation
tests from `experiments.stats` on the saved per-seed curves (no training).

In [ ]:
# ── 5.3  Speed: steps-to-threshold bars + cluster-permutation intervals ───────
from experiments.stats import paired_report, holm, cluster_perm_test

lc = load("exp5_learning_curves")
E5_EVAL = lc["eval_every"]
e5_rows = {m: lc["per_seed"][m] for m, _ in e5_methods}   # ragged per-seed eval curves

E5_SPEED_COMPS = [("full", "ablate_temporal", "full vs ablate_temporal", "C4"),
                  ("full", "ablate_spatial",  "full vs ablate_spatial",  "C5"),
                  ("ablate_temporal", "ablate_spatial", "temporal vs spatial", "C6")]

def _stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

# ── (A) Steps-to-threshold ────────────────────────────────────────────────────
def steps_to_threshold(r, theta, eval_every=E5_EVAL):
    r = np.asarray(r, float)
    above = np.where(r >= theta)[0]
    if len(above) == 0:
        return float("nan")
    i = int(above[0])
    if i == 0:
        return 0.0
    y0, y1 = r[i - 1], r[i]
    frac = 0.0 if y1 == y0 else (theta - y0) / (y1 - y0)
    return (i - 1 + min(max(frac, 0.0), 1.0)) * eval_every

s2t = {m: np.array([steps_to_threshold(r, E5_THETA) for r in e5_rows[m]]) for m, _ in e5_methods}
reports = [(lab, a, b, paired_report(s2t[a], s2t[b])) for a, b, lab, _ in E5_SPEED_COMPS]
e5_spd_padj = holm([r["p_perm"] for *_, r in reports])
print(f"Steps to reach acc>={E5_THETA:.2f} (paired sign-flip perm, Holm):")
for (lab, a, b, r), pa in zip(reports, e5_spd_padj):
    print(f"  {lab:26s} d_steps={r['mean_diff']:+7.0f}  dz={r['cohen_dz']:+5.2f}  "
          f"p={r['p_perm']:.4f}  Holm={pa:.4f}  {_stars(pa)}")

order = ["bptt", "full", "ablate_temporal", "ablate_spatial"]
short = {"bptt": "BPTT", "full": "full", "ablate_temporal": "ablate\ntemporal",
         "ablate_spatial": "ablate\nspatial"}
means = {m: float(np.nanmean(s2t[m])) for m in order}
sems  = {m: float(np.nanstd(s2t[m], ddof=1) / np.sqrt(np.sum(~np.isnan(s2t[m])))) for m in order}
figA, axA = plt.subplots(figsize=(7, 4.6)); xs = np.arange(len(order))
axA.bar(xs, [means[m] for m in order], yerr=[sems[m] for m in order], capsize=4,
        color=[col[m] for m in order])
axA.set_xticks(xs); axA.set_xticklabels([short[m] for m in order])
axA.set_ylabel(f"training steps to reach acc >= {E5_THETA:.2f}")
axA.set_title(f"Convergence speed (Adam): steps to {E5_THETA:.0%} accuracy")
y0 = max(means[m] + sems[m] for m in order)
for k, (a, b, lab, _) in enumerate(E5_SPEED_COMPS):
    xa, xb = order.index(a), order.index(b); y = y0 * (1.06 + 0.10 * k)
    axA.plot([xa, xa, xb, xb], [y - y0 * 0.02, y, y, y - y0 * 0.02], color="k", lw=1)
    axA.text((xa + xb) / 2, y, _stars(e5_spd_padj[k]), ha="center", va="bottom", fontsize=11)
axA.set_ylim(0, y0 * 1.45)
figA.tight_layout(); save_fig(figA, "exp5_speed_threshold"); plt.show()

# ── (B) Cluster-permutation significant intervals over training ───────────────
n_common = min(len(r) for m, _ in e5_methods for r in e5_rows[m])
A = {m: np.array([r[:n_common] for r in e5_rows[m]]) for m, _ in e5_methods}
t_axis = np.arange(n_common) * E5_EVAL

e5_sig_intervals = {}
print(f"\nCluster-permutation significant intervals (0-{t_axis[-1]} steps):")
for a, b, lab, _c in E5_SPEED_COMPS:
    cl = cluster_perm_test(A[a] - A[b])
    sig = [c for c in cl if c["p_cluster"] < 0.05]
    e5_sig_intervals[(a, b)] = sig
    spans = ", ".join(f"[{t_axis[c['t0']]}-{t_axis[c['t1']]}] (p={c['p_cluster']:.3f})"
                      for c in sig) or "none"
    print(f"  {lab:26s} {spans}")

figB, axB = plt.subplots(figsize=(7, 4.8))
for meth, lab in e5_methods:
    axB.plot(t_axis, A[meth].mean(0), "-o", ms=3, color=col[meth], label=lab)
axB.axhline(0.5, color="gray", ls="--", lw=0.8)
axB.set_xlabel("training step"); axB.set_ylabel("accuracy (held-out)")
axB.set_ylim(0.40, 1.02)
axB.set_title("Where convergence-speed gaps are significant (cluster permutation)")
ybar = 0.46
for k, (a, b, lab, cc) in enumerate(E5_SPEED_COMPS):
    y = ybar - 0.018 * k
    for c in e5_sig_intervals[(a, b)]:
        axB.plot([t_axis[c["t0"]], t_axis[c["t1"]]], [y, y], color=cc, lw=4, solid_capstyle="butt")
    axB.plot([], [], color=cc, lw=4, label=f"sig: {lab}")
axB.legend(fontsize=7, ncol=2, loc="lower right")
figB.tight_layout(); save_fig(figB, "exp5_speed_intervals"); plt.show()


## Push regenerated figures to GitHub
Run this once you're happy with the figures above.

In [ ]:
# ── Commit & push the regenerated Exp 5 figures ───────────────────────────────
import subprocess

EXP5_FIGS = ["exp5_gradient_credit", "exp5_credit_summary", "exp5_learning_curves",
             "exp5_speed_threshold", "exp5_speed_intervals"]
paths = [f"results/{n}.{ext}" for n in EXP5_FIGS for ext in ("svg", "pdf")]
COMMIT_MSG = "Re-plot Exp 5 figures from saved JSON"

def run(*cmd):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    return r

run("git", "add", *paths)
status = subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode
if status == 0:
    print("Nothing changed — figures are identical to what is committed.")
else:
    run("git", "commit", "-m", COMMIT_MSG)
    run("git", "push")
